# Lab 4: MongoDB Basics, Aggregation Pipelines and Query Optimization

Lab 3 used **MongoDB Atlas** as the data layer of an agent: the model wrote filters and aggregation pipelines, and the tools `run_find` and `run_aggregate` executed them. Whether these pipelines were *correct* and *efficient* was not examined. This laboratory studies the database side, following **Course 4** (*MongoDB: Aggregation Pipelines, Optimizare & Vector Search*). **Lab 5** continues with vector search (Course 4, Part III) and with a guardrail for the agent.

The laboratory is **self-contained**: students who did not complete the previous labs can follow it from Section 0 (creating the Atlas cluster) and from the section *MongoDB basics*.

1. **MongoDB basics.** The document model, the correspondence with relational terminology, and the CRUD operations with PyMongo (`insert`, `find`, `update`, `delete`).
2. **The aggregation pipeline** (Course 4, Part I). Documents flow through a sequence of stages (`$match`, `$group`, `$unwind`, `$lookup`, `$facet`, `$setWindowFields`, ...).
3. **Query plans, indexes and pipeline optimization** (Course 4, Part II). How the optimizer chooses a plan, how a plan is read with `explain()`, how compound indexes are designed with the **ESR** rule, and which pipeline rewrites the optimizer performs.

### Datasets

| Dataset | Where | Used for |
|---|---|---|
| `students` | `bdnv_lab` (created below) | The CRUD operations |
| `articles`, `authors` | `bdnv_lab` (created below) | The 8-document collection of the Course 4 handout, so that every result can be verified by hand |
| `sample_airbnb.listingsAndReviews` | Atlas sample data | Nested documents and arrays (`address`, `host`, `amenities`, `reviews`) |
| `sample_mflix.movies`, `sample_mflix.comments` | Atlas sample data | Joins (`$lookup`), window functions |
| `articles_big`, `authors_big`, copies of `movies`, `comments`, `listingsAndReviews` | `bdnv_lab` (generated in Part II) | Optimization experiments with indexes under our control |

All writes go to the scratch database **`bdnv_lab`**; the sample databases are only read.

### Suggested schedule (2 hours)

| Section | Time | |
|---|---|---|
| 0. Getting started with MongoDB Atlas | 15 min | **before the lab**, only if not done in Lab 3 |
| Setting up the environment | 5 min | |
| MongoDB basics and Exercise 1 | 25 min | |
| Part I: The aggregation pipeline | 25 min | |
| Exercises 2 and 3.1–3.2 | 15 min | in the lab |
| Part II: Query plans, indexes and optimization | 35 min | |
| Exercise 6 | 15 min | in the lab |
| Exercises 3.3–3.4, 4, 5 and 7 | | **homework** |

### Limits of the free tier (M0) relevant to this laboratory

| Limit on M0 | Consequence |
|---|---|
| 512 MB of storage | The synthetic collections are kept small (`N_BIG = 100_000`); do not increase the sizes. |
| `allowDiskUse` is ignored; in-memory sort limit of 32 MB | A blocking stage that exceeds its memory limit **fails** instead of spilling to disk. |
| No database profiler, no Performance Advisor | The profiler cell only reports that the command is unavailable. |

## 0. Getting started with MongoDB Atlas

*Students who completed Lab 3 already have a cluster with the sample data and can skip to "Setting up the environment". Steps 1–5 take about 15 minutes, most of it waiting for the cluster and the sample data; **complete them before the laboratory**.*

**MongoDB Atlas** is the managed cloud service of MongoDB. Its free tier (**M0**) is sufficient for this laboratory and also offers Atlas Vector Search, which is used in Lab 5. The labels of the Atlas interface change from time to time; if a button has a slightly different name, look for the equivalent option.

1. **Create an account** at https://www.mongodb.com/cloud/atlas/register (an e-mail address or a Google account). Atlas creates an *organization* and a *project* for you.
2. **Create a free cluster.** In the project, choose *Create* (or *Build a Database*), select the **M0 / Free** tier, a cloud provider and a region close to you (e.g. AWS, Frankfurt `eu-central-1`), keep the name `Cluster0` and confirm. The cluster is ready after a few minutes.
3. **Create a database user.** *Security → Database Access → Add New Database User*: authentication method *Password*, a username and a strong password (save them), built-in role **Read and write to any database**. This user is different from your Atlas login account; it is the one the notebook uses.
4. **Allow network access.** *Security → Network Access → Add IP Address*. Colab runs on Google machines whose IP address changes, so choose **Allow Access from Anywhere** (`0.0.0.0/0`). The cluster remains protected by the user and password; use a strong password and remove this rule at the end of the semester. When working only from your own computer, *Add Current IP Address* is sufficient.
5. **Load the sample data.** On the cluster card, open the menu **`...` → Load Sample Dataset** (in newer interfaces, the option may be offered when the cluster is created). The labs use the databases `sample_airbnb` and `sample_mflix`. Loading takes a few minutes; the databases appear under *Browse Collections* (or *Data Explorer*), where you can also inspect the documents.
6. **Copy the connection string.** *Connect → Drivers → Python*. The string has the form

   ```
   mongodb+srv://<username>:<password>@cluster0.xxxxx.mongodb.net/?retryWrites=true&w=majority&appName=Cluster0
   ```

   Replace `<password>` with the password of the database user (without the angle brackets). If the password contains special characters (`@`, `:`, `/`, `%`, ...), encode it with `urllib.parse.quote_plus(password)` or choose a password without such characters.
7. **Store the connection string as a secret, never in the notebook.**
   - **Colab:** the key icon in the left sidebar (*Secrets*) → *Add new secret*, name `MONGODB_URI`, value the connection string, and enable *Notebook access*.
   - **Locally (Jupyter / VS Code):** a `.env` file next to the notebook containing `MONGODB_URI=mongodb+srv://...`, as for the API keys in Lab 1. The `.env` file must be listed in `.gitignore`.

**Common errors.** `ServerSelectionTimeoutError`: the network access rule (step 4) is missing or the cluster is still being created. `bad auth : authentication failed`: wrong username/password, or the Atlas login password was used instead of the database user password. `not authorized ... to execute command insert`: the user has a read-only role (e.g. the user of Lab 3); use a read-write user.

## Setting up the environment

### Installing the dependencies

`pymongo` is the official MongoDB driver for Python; `python-dotenv` reads the `.env` file when the notebook runs outside Colab.

In [ ]:
!pip install -qU pymongo python-dotenv

### Connecting to the cluster

The connection string is read from a Colab secret, from a `.env` file or from an environment variable (Section 0, step 7); if none is found, it is requested interactively without being displayed. `MongoClient` opens a pool of connections to the cluster; `ping` verifies that the cluster can be reached and that the credentials are valid. From the client, a database is obtained with `client["name"]` and a collection with `database["name"]`; neither has to be created explicitly: they are created by the first write.

In [ ]:
import os
from getpass import getpass

from pymongo import MongoClient
from pymongo.errors import OperationFailure


def get_secret(name: str) -> str | None:
    # Read a secret from Colab, from a local .env file or from the environment.
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    return os.environ.get(name)


MONGODB_URI = get_secret("MONGODB_URI") or getpass("MongoDB Atlas connection string (read-write): ")

client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=15_000)
client.admin.command("ping")
print("Connected. Server version:", client.server_info()["version"])

airbnb = client["sample_airbnb"]["listingsAndReviews"]   # read only
mflix = client["sample_mflix"]                            # read only
lab = client["bdnv_lab"]                                  # scratch database: everything these labs create

### Checking the environment

The next cell verifies the prerequisites and reports what is missing, with a pointer to the step of Section 0 that fixes it. Run it again after fixing a problem.

In [ ]:
def check(label: str, ok: bool, fix: str = "") -> bool:
    print(f"[{'OK' if ok else '!!'}] {label}" + ("" if ok else f"\n       -> {fix}"))
    return ok


try:
    databases = client.list_database_names()
except OperationFailure:
    databases = [name for name in ("sample_airbnb", "sample_mflix") if client[name].list_collection_names()]

HAS_AIRBNB = check("sample_airbnb is loaded", "sample_airbnb" in databases,
                   "Load the sample dataset (Section 0, step 5).")
HAS_MFLIX = check("sample_mflix is loaded", "sample_mflix" in databases,
                  "Load the sample dataset (Section 0, step 5).")
try:
    lab["_write_check"].insert_one({"ok": True})
    lab["_write_check"].drop()
    CAN_WRITE = True
except OperationFailure:
    CAN_WRITE = False
check("the user can write (scratch database bdnv_lab)", CAN_WRITE,
      "Use a user with the role 'Read and write to any database' (Section 0, step 3).")

try:
    LAB3_SEARCH_INDEXES = [index["name"] for index in airbnb.list_search_indexes()]
    HAS_SEARCH = True
except OperationFailure:
    LAB3_SEARCH_INDEXES, HAS_SEARCH = [], False
check("Atlas Search / Vector Search is available", HAS_SEARCH,
      "Vector search requires an Atlas cluster (or a local Atlas deployment); it is used in Lab 5.")
if HAS_SEARCH:
    print(f"[--] vector index of Lab 3 on listingsAndReviews: "
          f"{'present' if 'vector_index' in LAB3_SEARCH_INDEXES else 'absent (not needed)'}")

### Helper functions

The following helpers are used throughout the notebook:

| Function | Purpose |
|---|---|
| `show(documents)` | Prints documents as compact JSON (`Decimal128`, `ObjectId` and dates as strings) |
| `stage_counts(collection, pipeline)` | Prints the number of documents that leave **each stage**, by running every prefix of the pipeline followed by `$count`. It reproduces the "how many documents leave each stage?" questions of the course handout. |
| `explain(collection, pipeline, ...)` | Runs the `explain` command for an aggregation pipeline or a `find` (with `filter`, `projection`, `sort`, `limit`, `hint`) at a given verbosity |
| `plan_summary(explanation)` | Extracts from the output of `explain` the stages of the winning plan, the indexes used, the metrics `nReturned`, `totalKeysExamined`, `totalDocsExamined`, the execution time and the pipeline stages that remain after the query layer |
| `timed(function)` | Median wall-clock time of a call in milliseconds. It includes the network round trip to Atlas, so it is useful for comparisons, not as an absolute measure. |

The output of `explain` differs between the classic execution engine and the **slot-based execution engine (SBE)**: with SBE, the plan is nested under `winningPlan.queryPlan`, and stages such as `$group` and `$lookup` appear as plan stages (`GROUP`, `EQ_LOOKUP`); with the classic engine, the query layer appears as a `$cursor` stage followed by the remaining pipeline stages. `plan_summary` handles both forms.

In [ ]:
import json
import time
from statistics import median


def show(documents, limit: int = 10) -> None:
    # Print documents as compact JSON.
    for i, document in enumerate(documents):
        if i == limit:
            print("...")
            break
        print(json.dumps(document, default=str, ensure_ascii=False))


def stage_counts(collection, pipeline: list[dict]) -> None:
    # Print the number of documents that leave each stage of a pipeline.
    for i in range(1, len(pipeline) + 1):
        result = list(collection.aggregate(pipeline[:i] + [{"$count": "n"}]))
        n = result[0]["n"] if result else 0
        print(f"({i}) {next(iter(pipeline[i - 1])):<18} -> {n} documents")


def timed(function, repeat: int = 3) -> float:
    # Median wall-clock time of function() in milliseconds.
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        function()
        times.append((time.perf_counter() - start) * 1000)
    return round(median(times), 1)


def explain(collection, pipeline: list[dict] | None = None, *, filter: dict | None = None,
            projection: dict | None = None, sort: dict | None = None, limit: int | None = None,
            hint=None, verbosity: str = "executionStats") -> dict:
    # Run the explain command for an aggregation pipeline or for a find().
    if pipeline is not None:
        command = {"aggregate": collection.name, "pipeline": pipeline, "cursor": {}}
    else:
        command = {"find": collection.name, "filter": filter or {}}
        if projection:
            command["projection"] = projection
        if sort:
            command["sort"] = sort
        if limit:
            command["limit"] = limit
    if hint:
        command["hint"] = hint
    return collection.database.command({"explain": command, "verbosity": verbosity})


def _walk_plan(node, stages: list, indexes: list) -> None:
    # Collect the stage names of a plan tree (root first) and the indexes it uses.
    if not isinstance(node, dict):
        return
    node = node.get("queryPlan", node)
    if isinstance(node.get("stage"), str):
        name = node["stage"]
        if "strategy" in node:                       # $lookup in SBE: EQ_LOOKUP(strategy)
            name += f"({node['strategy']})"
        stages.append(name)
        if "indexName" in node:
            indexes.append(node["indexName"])
    for key in ("inputStage", "outerStage", "innerStage"):
        _walk_plan(node.get(key), stages, indexes)
    for child in node.get("inputStages", []):
        _walk_plan(child, stages, indexes)


def _find_values(obj, key: str):
    # Yield every value stored under `key` anywhere in a nested document.
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == key:
                yield v
            yield from _find_values(v, key)
    elif isinstance(obj, list):
        for item in obj:
            yield from _find_values(item, key)


def plan_summary(explanation: dict, quiet: bool = False) -> dict:
    # Summarize the output of explain(): plan stages, indexes, metrics, remaining pipeline stages.
    stages = explanation.get("stages") or []
    has_cursor = bool(stages) and "$cursor" in stages[0]
    layer = stages[0]["$cursor"] if has_cursor else explanation
    plan, indexes = [], []
    _walk_plan(layer.get("queryPlanner", {}).get("winningPlan", {}), plan, indexes)
    stats = layer.get("executionStats", {})
    remaining = stages[1:] if has_cursor else stages
    pipeline_stages = []
    for stage in remaining:
        name = next(k for k in stage if k.startswith("$"))
        details = [f"out={stage['nReturned']}"] if "nReturned" in stage else []
        for key in ("strategy", "collectionScans", "indexesUsed", "totalDocsExamined", "usedDisk"):
            if key in stage and (key != "usedDisk" or stage[key]):
                details.append(f"{key}={stage[key]}")
        pipeline_stages.append(f"{name}({', '.join(details)})" if details else name)
    summary = {
        "plan": " <- ".join(plan) or "-",
        "indexes": indexes,
        "nReturned": stats.get("nReturned"),
        "keysExamined": stats.get("totalKeysExamined"),
        "docsExamined": stats.get("totalDocsExamined"),
        "timeMs": stats.get("executionTimeMillis"),
        "pipeline": pipeline_stages,
        "usedDisk": any(v is True for v in _find_values(explanation, "usedDisk")),
    }
    if not quiet:
        print(f"plan         : {summary['plan']}")
        print(f"indexes      : {summary['indexes'] or 'none'}")
        if stats:
            print(f"nReturned    : {summary['nReturned']} | keysExamined: {summary['keysExamined']} "
                  f"| docsExamined: {summary['docsExamined']} | executionTimeMillis: {summary['timeMs']}")
        if pipeline_stages:
            print(f"after query  : {' -> '.join(pipeline_stages)}")
        if summary["usedDisk"]:
            print("usedDisk     : True (a blocking stage spilled to disk)")
    return summary


def summary_table(rows: dict[str, dict]) -> None:
    # Print several plan summaries side by side.
    print(f"{'variant':<28}{'plan':<44}{'nReturned':>10}{'keys':>9}{'docs':>9}{'ms':>6}")
    for label, s in rows.items():
        print(f"{label:<28}{s['plan'][:43]:<44}{str(s['nReturned']):>10}{str(s['keysExamined']):>9}"
              f"{str(s['docsExamined']):>9}{str(s['timeMs']):>6}")

## MongoDB basics

*Students who used MongoDB before can skim this section and go directly to Exercise 1.*

### The document model

MongoDB is a **document database**. A record is a **document**: a set of field–value pairs, similar to a JSON object, stored in the binary format **BSON**. A value can be a scalar, an **embedded document** or an **array**, so that related data can be stored together in one document instead of being spread over several tables. Documents are grouped in **collections**, and collections in **databases**. A collection does not impose a schema: two documents of the same collection may have different fields, although in practice the documents of a collection have a similar structure.

| Relational | MongoDB |
|---|---|
| database | database |
| table | collection |
| row | document |
| column | field |
| primary key | `_id` (mandatory, unique; generated as an `ObjectId` if missing) |
| foreign key + `JOIN` | embedded document / array, or a reference + `$lookup` |
| `SELECT ... WHERE ... ORDER BY ... LIMIT` | `find(filter, projection).sort(...).limit(...)` |
| `GROUP BY`, views, complex queries | `aggregate(pipeline)` |
| index | index (B-tree), also on nested fields and arrays |

The BSON types that appear in the sample data include `String`, `Int32`/`Int64`, `Double`, **`Decimal128`** (exact decimal numbers, used for prices), **`Date`**, **`ObjectId`**, `Boolean`, `null`, arrays and embedded documents. In PyMongo, a document is a Python `dict`, an array a `list`, a `Date` a `datetime`, and an `ObjectId` an instance of `bson.ObjectId`.

**Embedding vs. referencing.** In `sample_airbnb`, the host (`host`), the address (`address`) and the reviews (`reviews`) are **embedded** in each listing: one read returns everything that a listing page shows. In `sample_mflix`, the comments are stored in a separate collection and **reference** their movie through `movie_id`, since a movie may receive an unbounded number of comments and they are usually read separately. The choice of model determines which queries need a join.

The next cell prints the structure of a listing: the top-level fields, their BSON/Python types and, for embedded documents, the nested fields.

In [ ]:
def describe(document: dict, indent: int = 0, max_fields: int = 40) -> None:
    # Print the fields of a document with their types, recursively for embedded documents.
    for i, (key, value) in enumerate(document.items()):
        if i == max_fields:
            print(" " * indent + "...")
            break
        kind = type(value).__name__
        if isinstance(value, list):
            inner = type(value[0]).__name__ if value else "empty"
            print(" " * indent + f"{key}: list[{inner}] ({len(value)} elements)")
            if value and isinstance(value[0], dict):
                describe(value[0], indent + 4, max_fields=8)
        elif isinstance(value, dict):
            print(" " * indent + f"{key}: embedded document")
            describe(value, indent + 4, max_fields=12)
        else:
            print(" " * indent + f"{key}: {kind} = {repr(value)[:60]}")


listing = airbnb.find_one({"address.market": "Porto", "number_of_reviews": {"$gte": 1}},
                          {"images": 0})
describe(listing)

### CRUD operations with PyMongo

The four basic operations are illustrated on a small collection `students` in the scratch database:

| Operation | PyMongo | SQL |
|---|---|---|
| Create | `insert_one(doc)`, `insert_many(docs)` | `INSERT` |
| Read | `find_one(filter)`, `find(filter, projection)`, `count_documents(filter)`, `distinct(field)` | `SELECT` |
| Update | `update_one(filter, update)`, `update_many(...)`, `replace_one(...)` | `UPDATE` |
| Delete | `delete_one(filter)`, `delete_many(filter)` | `DELETE` |

An update is expressed with **update operators**: `$set` (set a field), `$unset` (remove a field), `$inc` (increment), `$push` / `$pull` (add to / remove from an array), `$addToSet` (add if absent). Without an operator, `replace_one` replaces the whole document. `find` returns a **cursor**, which fetches the documents in batches only when it is iterated.

In [ ]:
students = lab["students"]
students.delete_many({})                                # start from an empty collection

# Create
result = students.insert_one({"name": "Ana", "year": 3, "grades": [9, 10],
                              "address": {"city": "București"}})
print("Generated _id:", result.inserted_id)
students.insert_many([
    {"name": "Mihai", "year": 3, "grades": [7, 8, 10], "address": {"city": "Cluj"}},
    {"name": "Elena", "year": 2, "grades": [10], "address": {"city": "București"}, "erasmus": True},
    {"name": "Dan", "year": 1, "grades": [], "address": {"city": "Iași"}},
])

# Read
print("\nYear 3:")
show(students.find({"year": 3}, {"_id": 0, "name": 1, "grades": 1}))
print("\nAt least one grade 10 (a condition on an array matches any element):")
show(students.find({"grades": 10}, {"_id": 0, "name": 1}))
print("\nLiving in București (dot notation for embedded fields):")
show(students.find({"address.city": "București"}, {"_id": 0, "name": 1}))
print("\nWith the field `erasmus`:", [s["name"] for s in students.find({"erasmus": {"$exists": True}})])

# Update
students.update_one({"name": "Ana"}, {"$set": {"year": 4}, "$push": {"grades": 8}})
students.update_many({}, {"$inc": {"credits": 30}})
print("\nAfter the updates:")
show(students.find({}, {"_id": 0}))

# Delete
students.delete_one({"name": "Dan"})
print("\nDocuments left:", students.count_documents({}))

### Queries with `find()`

The **filter** of `find` is a document of conditions; several conditions in the same document are combined with AND. The most common **query operators**:

| Operator | Meaning | Example |
|---|---|---|
| `$eq`, `$ne` | equal, not equal | `{"room_type": {"$ne": "Shared room"}}` |
| `$gt`, `$gte`, `$lt`, `$lte` | comparisons | `{"bedrooms": {"$gte": 3}}` |
| `$in`, `$nin` | belongs / does not belong to a list | `{"address.market": {"$in": ["Porto", "Barcelona"]}}` |
| `$and`, `$or`, `$nor`, `$not` | logical operators | `{"$or": [{"bedrooms": 0}, {"beds": 0}]}` |
| `$exists`, `$type` | the field exists / has a BSON type | `{"year": {"$type": "number"}}` |
| `$regex` | regular expression | `{"name": {"$regex": "^Loft", "$options": "i"}}` |
| `$all`, `$size`, `$elemMatch` | conditions on arrays | `{"amenities": {"$all": ["Wifi", "Pool"]}}` |

The **projection** (second argument) selects the returned fields: `{"name": 1, "price": 1, "_id": 0}`. The cursor methods `sort`, `skip` and `limit` correspond to `ORDER BY`, `OFFSET` and `LIMIT`.

In [ ]:
print("The 5 cheapest listings in Porto with at least 3 bedrooms:")
show(airbnb.find({"address.market": "Porto", "bedrooms": {"$gte": 3}},
                 {"_id": 0, "name": 1, "bedrooms": 1, "price": 1})
           .sort("price", 1).limit(5))

print("\nListings in Portugal with Wifi and a pool:")
show(airbnb.find({"address.country_code": "PT", "amenities": {"$all": ["Wifi", "Pool"]}},
                 {"_id": 0, "name": 1}).limit(3))

print("\nLofts or listings for at least 10 guests in Barcelona:")
show(airbnb.find({"address.market": "Barcelona",
                  "$or": [{"property_type": "Loft"}, {"accommodates": {"$gte": 10}}]},
                 {"_id": 0, "name": 1, "property_type": 1, "accommodates": 1}).limit(5))

print("\nMarkets:", airbnb.distinct("address.market"))
print("Listings without reviews:", airbnb.count_documents({"reviews": {"$size": 0}}))

### From `find()` to `aggregate()`

`find` filters, projects, sorts and limits the documents of **one** collection, and returns them as they are stored. Everything else (grouping and counting, computing new fields, unwinding arrays, joining collections, statistics) is expressed with the **aggregation pipeline**, which is the subject of Part I. In fact, the first stages of a pipeline (`$match`, `$sort`, `$limit`, `$project`) are executed in the same way as a `find`, with the same indexes (Part II).

#### Exercise 1: Queries with `find()` *(in the lab, ~10 min)*

*Objective: to write filters on nested fields and arrays, and to relate them to SQL.*

1. The listings in Hong Kong (`address.market`) with at least 2 bedrooms and a price below 500 (local currency, HKD): name and price, sorted by price, the first 5.
2. The number of listings in Montreal whose `amenities` contain both `"Kitchen"` and `"Washer"`.
3. The listings in Sydney whose host is a superhost (`host.host_is_superhost`) and whose rating (`review_scores.review_scores_rating`) is at least 98. Return only the name and the host's name.
4. In `sample_mflix.movies`: the movies with `"Tom Hanks"` in `cast`, released after 2000 (`year`), with title and year, sorted by year.
5. In `students`: add yourself, add a grade with `$push`, change the city with `$set`, then delete the document. After each step, verify with `find`.
6. Write the SQL equivalent of queries 1 and 3. Which SQL construct would query 2 require if `amenities` were stored in a separate table?

In [ ]:
# Exercise 1

### The course dataset: `articles` and `authors`

The collection `articles` contains the eight documents of the Course 4 handout. The embeddings have only **four dimensions**, so that dot products can be computed by hand, and all vectors are **L2-normalized** ($\lVert e \rVert = 1$), as in Step 7 of Course 2. A real model such as `all-MiniLM-L6-v2` produces vectors with $d = 384$, and `nomic-embed-text` (Lab 3) vectors with $d = 768$. Each article also references an author in the collection `authors` (field `authorId`), as in the slides on `$lookup`. The query vector, obtained "with the same model", is $q = [0.5, 0.7, 0.5, 0.1]$.

| `_id` | `title` | `lang` | `year` | `tags` | `embedding` |
|---|---|---|---|---|---|
| A1 | Introducere în MongoDB | ro | 2023 | mongodb, nosql | [1.0, 0.0, 0.0, 0.0] |
| A2 | Aggregation pipeline pas cu pas | ro | 2024 | mongodb, pipeline | [0.6, 0.8, 0.0, 0.0] |
| A3 | Indecși B-tree și regula ESR | en | 2024 | index, mongodb | [0.5, 0.5, 0.5, 0.5] |
| A4 | Embeddings pentru text | ro | 2025 | nlp, vector | [0.0, 0.6, 0.8, 0.0] |
| A5 | HNSW explicat | en | 2025 | vector, index | [0.0, 0.0, 0.6, 0.8] |
| A6 | Tokenizare BPE | ro | 2025 | nlp | [0.0, 0.0, 0.0, 1.0] |
| A7 | Căutare hibridă | ro | 2026 | vector, search | [0.0, 0.8, 0.0, 0.6] |
| A8 | Quantization pentru vectori | en | 2026 | vector | [0.8, 0.0, 0.6, 0.0] |

The document also stores the name of the embedding model (`emb_model`), as recommended in the course: a query must be embedded with the same model as the stored documents.

In [ ]:
import math

ARTICLES = [
    ("A1", "Introducere în MongoDB",          "ro", 2023, ["mongodb", "nosql"],    [1.0, 0.0, 0.0, 0.0], "au1"),
    ("A2", "Aggregation pipeline pas cu pas", "ro", 2024, ["mongodb", "pipeline"], [0.6, 0.8, 0.0, 0.0], "au1"),
    ("A3", "Indecși B-tree și regula ESR",    "en", 2024, ["index", "mongodb"],    [0.5, 0.5, 0.5, 0.5], "au2"),
    ("A4", "Embeddings pentru text",          "ro", 2025, ["nlp", "vector"],       [0.0, 0.6, 0.8, 0.0], "au3"),
    ("A5", "HNSW explicat",                   "en", 2025, ["vector", "index"],     [0.0, 0.0, 0.6, 0.8], "au2"),
    ("A6", "Tokenizare BPE",                  "ro", 2025, ["nlp"],                 [0.0, 0.0, 0.0, 1.0], "au3"),
    ("A7", "Căutare hibridă",                 "ro", 2026, ["vector", "search"],    [0.0, 0.8, 0.0, 0.6], "au4"),
    ("A8", "Quantization pentru vectori",     "en", 2026, ["vector"],              [0.8, 0.0, 0.6, 0.0], "au4"),
]
AUTHORS = [
    {"_id": "au1", "name": "Ana Ionescu",    "country": "RO"},
    {"_id": "au2", "name": "John Miller",    "country": "US"},
    {"_id": "au3", "name": "Elena Dumitru",  "country": "RO"},
    {"_id": "au4", "name": "Lena Schmidt",   "country": "DE"},
]
q = [0.5, 0.7, 0.5, 0.1]

articles = lab["articles"]
authors = lab["authors"]
articles.delete_many({})
authors.delete_many({})
articles.insert_many([
    {"_id": _id, "title": title, "lang": lang, "year": year, "tags": tags,
     "text": f"{title}. Text demonstrativ pentru cursul BDNV.", "authorId": author,
     "embedding": embedding, "emb_model": "course-demo-4d", "emb_norm": True}
    for _id, title, lang, year, tags, embedding, author in ARTICLES
])
authors.insert_many(AUTHORS)

for document in articles.find({}, {"title": 1, "embedding": 1}):
    norm = math.sqrt(sum(x * x for x in document["embedding"]))
    print(f"{document['_id']}  ||e|| = {norm:.3f}  {document['title']}")
print(f"||q|| = {math.sqrt(sum(x * x for x in q)):.3f}")

## Part I: The aggregation pipeline

An **aggregation pipeline** is a list of stages. Each stage receives a stream of documents and emits another stream, like the commands of a Unix pipe (`cat log | grep ERROR | sort | head`). The pipeline runs on the server, close to the data, and only the result is transferred to the client. Three properties follow from this model (Course 4, slide 8):

- **The order of the stages matters.** The same stages in a different order yield a different result or the same result at a different cost.
- **A stage may occur several times**, e.g. `$match` before and after `$group`.
- **A stage only sees the output of the previous stage.** After `$group`, the original documents no longer exist; after `$unwind`, each document represents one element of an array.

The stages used most often and their SQL counterparts (Course 4, slide 11):

| Stage | Role | SQL counterpart |
|---|---|---|
| `$match` | filters documents | `WHERE` / `HAVING` |
| `$project`, `$set`, `$unset` | shape the fields | `SELECT` list |
| `$group` | grouping with accumulators | `GROUP BY` |
| `$sort`, `$limit`, `$skip` | ordering, pagination | `ORDER BY`, `LIMIT`, `OFFSET` |
| `$unwind` | one document per array element | `UNNEST` |
| `$lookup` | left outer join | `LEFT JOIN` |
| `$facet`, `$bucket` | several aggregations of the same input | `GROUPING SETS` (approx.) |
| `$setWindowFields` | window functions | `OVER (PARTITION BY ...)` |
| `$out`, `$merge` | write the result to a collection | `INSERT ... SELECT`, `MERGE` |
| `$unionWith` | concatenates collections | `UNION ALL` |
| `$vectorSearch` | top-k nearest neighbors (ANN / ENN) | - |

In PyMongo, a pipeline is a Python list of dictionaries. Field values are referenced as `"$field"` (with dot notation for nested fields, e.g. `"$address.market"`), and variables as `"$$name"`.

### A first pipeline: the worked example of the handout

*Which tags are the most frequent in the Romanian articles?* The pipeline filters, unwinds the `tags` array, groups by tag and sorts. `stage_counts` shows how many documents leave each stage.

In [ ]:
tags_ro = [
    {"$match": {"lang": "ro"}},                       # (1) WHERE lang = 'ro'
    {"$unwind": "$tags"},                             # (2) one document per tag
    {"$group": {"_id": "$tags", "n": {"$sum": 1}}},   # (3) GROUP BY tag
    {"$sort": {"n": -1, "_id": 1}},                   # (4) ORDER BY n DESC, tag
]
stage_counts(articles, tags_ro)
print()
show(articles.aggregate(tags_ro))

**Points to observe.**

1. `$match` keeps the five Romanian articles (A1, A2, A4, A6, A7). `$unwind` produces nine documents, one per tag ($2 + 2 + 2 + 1 + 2$), and `$group` reduces them to six groups.
2. The sort key `{"n": -1, "_id": 1}` breaks ties between tags with the same frequency. Without a tie-breaker, the order of equal keys is not guaranteed, and a test that compares results would be unstable.
3. Only stage (1) can use an index of the collection (e.g. `{lang: 1}`): after `$unwind`, the documents in the stream no longer correspond to the entries of any index. This observation is the starting point of Part II.

### `$match` before and after `$group`: `WHERE` vs. `HAVING`

On `sample_airbnb`: *for the listings with at least 10 reviews, which markets have at least 250 such listings, and what is their average rating?* The first `$match` filters documents (`WHERE`); the second filters groups (`HAVING`), because it refers to the field `n`, which exists only after `$group`.

```sql
SELECT market, COUNT(*) AS n, AVG(rating) AS avgRating
FROM   listings
WHERE  number_of_reviews >= 10
GROUP  BY market
HAVING COUNT(*) >= 250
ORDER  BY avgRating DESC;
```

In [ ]:
markets_rating = [
    {"$match": {"number_of_reviews": {"$gte": 10}}},                       # WHERE
    {"$group": {"_id": "$address.market",
                "n": {"$sum": 1},
                "avgRating": {"$avg": "$review_scores.review_scores_rating"}}},
    {"$match": {"n": {"$gte": 250}}},                                       # HAVING
    {"$sort": {"avgRating": -1}},
    {"$project": {"n": 1, "avgRating": {"$round": ["$avgRating", 2]}}},
]
stage_counts(airbnb, markets_rating)
print()
show(airbnb.aggregate(markets_rating))

### `$group` and accumulators

The `_id` of `$group` is the grouping key (a single field, a compound document or an expression). The other fields are **accumulators** (Course 4, slide 12): `$sum`, `$avg`, `$min`, `$max`, `$push`, `$addToSet`, `$first`, `$last`, and the bounded `$topN` / `$firstN`, which keep only `n` values.

Two properties of the dataset matter here:

- `price` is stored as `Decimal128`; `$toDouble` converts it for arithmetic.
- Prices are in the **local currency of each market** (as noted in Lab 3). Averages are therefore comparable within a market, not across markets.

`$group` is a **blocking** stage: it consumes its whole input before emitting the first group, and it is limited to 100 MB of memory (Part II).

In [ ]:
market_stats = [
    {"$group": {
        "_id": "$address.market",
        "listings": {"$sum": 1},
        "avgPrice": {"$avg": {"$toDouble": "$price"}},          # local currency!
        "maxGuests": {"$max": "$accommodates"},
        "propertyTypes": {"$addToSet": "$property_type"},
        "bestRated": {"$topN": {"n": 3,
                                "sortBy": {"review_scores.review_scores_rating": -1, "number_of_reviews": -1},
                                "output": "$name"}},
    }},
    {"$set": {"avgPrice": {"$round": ["$avgPrice", 1]},
              "propertyTypes": {"$size": "$propertyTypes"}}},
    {"$sort": {"listings": -1}},
]
show(airbnb.aggregate(market_stats))

### `$unwind`: working with arrays

`$unwind` emits one document for every element of an array (Course 4, slide 13). The number of documents is multiplied by the size of the arrays, and documents whose array is missing or empty **disappear** unless `preserveNullAndEmptyArrays: true` is specified. When only the size of an array is needed, `$size` computes it without unwinding.

*The 10 most frequent amenities in Porto:*

In [ ]:
porto = {"$match": {"address.market": "Porto"}}

top_amenities = [
    porto,
    {"$unwind": "$amenities"},
    {"$group": {"_id": "$amenities", "n": {"$sum": 1}}},
    {"$sort": {"n": -1, "_id": 1}},
    {"$limit": 10},
]
stage_counts(airbnb, top_amenities)
print()
show(airbnb.aggregate(top_amenities))

# Only the size is needed: $size, without $unwind
print()
show(airbnb.aggregate([porto, {"$group": {"_id": None, "avgAmenities": {"$avg": {"$size": "$amenities"}}}}]))

In [ ]:
# Listings without reviews disappear after $unwind, unless preserveNullAndEmptyArrays is set
def count_listings(unwind_stage) -> int:
    result = list(airbnb.aggregate([porto, unwind_stage, {"$group": {"_id": "$_id"}}, {"$count": "n"}]))
    return result[0]["n"] if result else 0

print("Porto listings                   :", airbnb.count_documents({"address.market": "Porto"}))
print("after $unwind of reviews         :", count_listings({"$unwind": "$reviews"}))
print("with preserveNullAndEmptyArrays  :", count_listings({"$unwind": {"path": "$reviews",
                                                                          "preserveNullAndEmptyArrays": True}}))

# Reviews of Porto listings per year: the embedded array is unwound and grouped by an expression
show(airbnb.aggregate([
    porto,
    {"$unwind": "$reviews"},
    {"$group": {"_id": {"$year": "$reviews.date"}, "reviews": {"$sum": 1}}},
    {"$sort": {"_id": 1}},
]))

### `$lookup`: joins between collections

`$lookup` performs a **left outer join** with another collection *of the same database* (Course 4, slide 14). It has two forms:

1. **Equality match** (`localField`, `foreignField`): the documents of the foreign collection whose `foreignField` equals the `localField` of the input document.
2. **Sub-pipeline** (`let`, `pipeline`): an arbitrary pipeline on the foreign collection, which can filter with `$expr`, sort and limit; variables defined in `let` are referenced as `$$name`.

The result is always an **array** (field `as`), which is often followed by `$unwind`. In `sample_mflix`, each comment references its movie through `movie_id`.

*The five most commented movies.* The comments are grouped **first**, and the join is performed only for the five resulting documents. Joining every comment with its movie and grouping afterwards would compute the same result with about 50,000 joins instead of five; Part II returns to this order of operations.

In [ ]:
most_commented = [
    {"$group": {"_id": "$movie_id", "comments": {"$sum": 1}}},
    {"$sort": {"comments": -1, "_id": 1}},
    {"$limit": 5},
    {"$lookup": {"from": "movies", "localField": "_id", "foreignField": "_id", "as": "movie"}},  # 5 joins
    {"$unwind": "$movie"},
    {"$project": {"_id": 1, "title": "$movie.title", "year": "$movie.year", "comments": 1}},
]
top_movies = list(mflix["comments"].aggregate(most_commented))
show(top_movies)

In [ ]:
# Sub-pipeline form: the three most recent comments of each of these movies
latest_comments = [
    {"$match": {"_id": {"$in": [m["_id"] for m in top_movies[:3]]}}},
    {"$lookup": {
        "from": "comments",
        "let": {"movieId": "$_id"},
        "pipeline": [
            {"$match": {"$expr": {"$eq": ["$movie_id", "$$movieId"]}}},
            {"$sort": {"date": -1}},
            {"$limit": 3},
            {"$project": {"_id": 0, "name": 1, "date": 1, "text": {"$substrCP": ["$text", 0, 40]}}},
        ],
        "as": "latestComments",
    }},
    {"$project": {"_id": 0, "title": 1, "latestComments": 1}},
]
for movie in mflix["movies"].aggregate(latest_comments):
    print(movie["title"])
    show(movie["latestComments"])

### `$facet` and `$bucket`

`$facet` runs several sub-pipelines on the **same input** and returns a single document with one array per sub-pipeline (Course 4, slide 15). This is the typical query behind the filters of a search page: the results together with counts by category and by price range. `$bucket` groups documents into ranges defined by `boundaries`; documents outside the ranges go to the `default` bucket.

Two properties to remember: the sub-pipelines of `$facet` **cannot use indexes**, so the input should be reduced by a `$match` placed before `$facet`; and the output is a single document, subject to the 16 MB limit.

In [ ]:
porto_facets = [
    porto,                                                        # indexable filter BEFORE $facet
    {"$facet": {
        "byRoomType": [{"$sortByCount": "$room_type"}],
        "byPrice": [{"$bucket": {
            "groupBy": {"$toDouble": "$price"},
            "boundaries": [0, 50, 100, 200, 500],
            "default": "500+",
            "output": {"n": {"$sum": 1}, "avgRating": {"$avg": "$review_scores.review_scores_rating"}},
        }}],
        "topPropertyTypes": [{"$sortByCount": "$property_type"}, {"$limit": 3}],
    }},
]
result = next(airbnb.aggregate(porto_facets))
for facet, values in result.items():
    print(facet)
    show(values)

### `$setWindowFields`: window functions

`$setWindowFields` computes, for each document, a value over a **window** of neighboring documents, defined by `partitionBy`, `sortBy` and a window (`documents: [-4, 0]` = the current document and the four before it). Unlike `$group`, it does **not** reduce the number of documents (Course 4, slide 16); it corresponds to `OVER (PARTITION BY ... ORDER BY ... ROWS ...)` in SQL. Other window operators include `$rank`, `$denseRank`, `$shift`, `$derivative` and `$expMovingAvg`.

*For each year between 1980 and 2015: the number of movies, their average IMDb rating, the 5-year moving average of this rating, the cumulative number of movies and the difference from the previous year.* Some fields of `sample_mflix` contain values of mixed types (e.g. a few years stored as strings), so the first `$match` keeps only numeric values with `$type`.

In [ ]:
movies_per_year = [
    {"$match": {"type": "movie", "year": {"$type": "number", "$gte": 1980, "$lte": 2015},
                "imdb.rating": {"$type": "number"}}},
    {"$group": {"_id": "$year", "movies": {"$sum": 1}, "avgRating": {"$avg": "$imdb.rating"}}},
    {"$setWindowFields": {
        "sortBy": {"_id": 1},
        "output": {
            "avgRating5y": {"$avg": "$avgRating", "window": {"documents": [-4, 0]}},
            "moviesCumulative": {"$sum": "$movies", "window": {"documents": ["unbounded", "current"]}},
            "moviesPrevYear": {"$shift": {"output": "$movies", "by": -1}},
        },
    }},
    {"$project": {"movies": 1, "moviesCumulative": 1,
                  "change": {"$subtract": ["$movies", "$moviesPrevYear"]},
                  "avgRating": {"$round": ["$avgRating", 2]},
                  "avgRating5y": {"$round": ["$avgRating5y", 2]}}},
    {"$sort": {"_id": 1}},
]
show(mflix["movies"].aggregate(movies_per_year), limit=36)

### Exercises for Part I

#### Exercise 2: Tracing a pipeline (handout, Exercise 1) *(in the lab, ~5 min)*

*Objective: to predict the output of each stage before running the pipeline.*

```python
ex2 = [
    {"$match": {"year": {"$gte": 2025}}},
    {"$group": {"_id": "$lang", "n": {"$sum": 1}, "titles": {"$push": "$title"}}},
    {"$sort": {"n": -1}},
]
```

1. Using the table of the dataset, write down how many documents leave each stage and the final result. Only then verify with `stage_counts(articles, ex2)` and `articles.aggregate(ex2)`.
2. Write the equivalent SQL query (without the `titles` column).
3. Which stage can use an index `{year: 1}`? Why can the other stages not use it?
4. Add a stage that keeps only the languages with at least three articles. Where must it be placed, and which SQL clause does it correspond to?

#### Exercise 3: Analytics on `sample_airbnb`

*Objective: to combine `$match`, `$group`, `$unwind`, `$facet` and expressions on nested documents. Items 1–2 in the lab (~10 min), items 3–4 as homework.*

1. For each market and each value of `bedrooms` (1, 2 and 3 only), compute the number of listings and the average price per night. Keep only the groups with at least 20 listings and sort them by market and number of bedrooms. Use a compound grouping key.
2. Compare the 10 most frequent amenities of the listings with `review_scores.review_scores_rating >= 95` with those of all listings in the same market. Which amenities appear only in one of the two lists? (Hint: two pipelines, or one `$facet`.)
3. Using the embedded `reviews` array, find the 5 reviewers (`reviews.reviewer_name`) with the most reviews in Barcelona, together with the number of distinct listings they reviewed (`$addToSet` followed by `$size`).
4. For each `room_type`, compute the distribution of prices in Barcelona with `$bucket`, and the cheapest three listings with `$bottomN`/`$topN`, in a single `$facet` stage.

#### Exercise 4: Joins and window functions on `sample_mflix` *(homework)*

*Objective: to use `$lookup` and `$setWindowFields`, and to reason about the order of operations.*

1. Find the 10 directors (`directors` is an array) with at least five movies, ordered by the average IMDb rating of their movies. Show the number of movies and the best-rated title of each director (`$topN`).
2. Write two pipelines that return the 10 most commented movies of the 1990s with their titles: (a) starting from `movies`, with a `$lookup` to `comments` for every movie and `$size` of the result; (b) starting from `comments`, grouping first, with a `$lookup` only at the end. Verify that the results are identical, compare their execution times with `timed` and explain the difference with `explain()` (Part II).
3. For each genre (after `$unwind` of `genres`), rank the years 2000–2015 by the number of movies with `$rank` inside `$setWindowFields` (`partitionBy: "$_id.genre"`). Return the best year of each genre.

In [ ]:
# Starter cell for Exercise 2
ex2 = [
    {"$match": {"year": {"$gte": 2025}}},
    {"$group": {"_id": "$lang", "n": {"$sum": 1}, "titles": {"$push": "$title"}}},
    {"$sort": {"n": -1}},
]
# Write your predictions first, then uncomment:
# stage_counts(articles, ex2)
# show(articles.aggregate(ex2))

## Part II: Query plans, indexes and pipeline optimization

A **query plan** is the strategy that MongoDB uses to execute a query: which index to scan (if any), in which order, and which stages to apply afterwards (Course 4, slides 19–22). The **query optimizer** generates candidate plans from the available indexes. In the classic approach (*multi-planning*), the candidates run for a short trial period and the most productive one becomes the **winning plan**. From MongoDB 8.3, a **cost-based ranker** (CBR), which estimates the cost of each candidate from statistics instead of running it, is the default for eligible queries. The winning plan is stored in a **plan cache** per **query shape**: the structure of a query without its values (`{name: "John", age: 30}` and `{name: "Jane", age: 25}` share the shape `{name: 1, age: 1}`).

`explain()` reports the plan. Its verbosity determines how much is reported:

| Verbosity | Content |
|---|---|
| `queryPlanner` | The winning plan (and rejected plans); the query is **not executed** |
| `executionStats` | The winning plan and the statistics of its execution |
| `allPlansExecution` | Additionally, the statistics of the trial runs of all candidate plans |

The stages and metrics to look for (Course 4, slide 22):

| Plan stage | Meaning | | Metric | Meaning |
|---|---|---|---|---|
| `COLLSCAN` | reads the whole collection | | `nReturned` | documents returned |
| `IXSCAN` | scans a range of an index | | `totalKeysExamined` | index keys read |
| `FETCH` | reads the full documents | | `totalDocsExamined` | documents read |
| `SORT` | sorts in memory | | `executionTimeMillis` | server-side execution time |
| `PROJECTION_COVERED` | answers from the index alone | | `usedDisk` | a stage spilled to disk |

**The golden rule:** in a well-supported query, `totalKeysExamined ≈ totalDocsExamined ≈ nReturned`. A large ratio between the documents examined and the documents returned indicates that a suitable index is missing.

### Collections at scale

The effect of an index is invisible on eight documents. The next cell generates the collection `articles_big`, with the structure of `articles` and `N_BIG` documents, and `authors_big` with 1,000 authors. The documents are generated **on the server** by a pipeline: `$documents` creates a single document containing the numbers $0, \dots, N-1$, `$unwind` turns them into $N$ documents, `$rand` draws the field values and `$out` writes the result. No data is transferred from the notebook. The cell also copies `sample_mflix.movies` (without the plot fields) and `sample_mflix.comments` into `bdnv_lab`, so that the indexes of these collections are under our control.

In [ ]:
N_BIG = 100_000            # keep this value on M0 (512 MB of storage)
N_AUTHORS = 1_000
LANGS = ["ro", "en", "fr", "de", "it"]
TAGS = ["mongodb", "nosql", "vector", "index", "nlp", "pipeline", "search", "rag"]
LOREM = ("Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt "
         "ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation.")


def random_int(low: int, high: int) -> dict:
    # Expression: a random integer in [low, high) (a 64-bit integer for large ranges).
    convert = "$toInt" if high - low < 2**31 else "$toLong"
    return {convert: {"$add": [low, {"$floor": {"$multiply": [{"$rand": {}}, high - low]}}]}}


def random_choice(values: list) -> dict:
    # Expression: a random element of `values`.
    return {"$arrayElemAt": [values, random_int(0, len(values))]}


lab["articles_big"].drop()
lab.aggregate([
    {"$documents": [{"i": {"$range": [0, N_BIG]}}]},
    {"$unwind": "$i"},
    {"$project": {
        "_id": "$i",
        "title": {"$concat": ["Article ", {"$toString": "$i"}]},
        "lang": random_choice(LANGS),
        "year": random_int(2015, 2027),
        "tags": {"$setUnion": [[random_choice(TAGS), random_choice(TAGS)]]},
        "authorId": random_int(0, N_AUTHORS),
        "status": random_choice(["published", "published", "published", "draft"]),
        "text": {"$substrCP": [LOREM, 0, random_int(40, 180)]},
    }},
    {"$out": "articles_big"},
])

lab["authors_big"].drop()
lab.aggregate([
    {"$documents": [{"i": {"$range": [0, N_AUTHORS]}}]},
    {"$unwind": "$i"},
    {"$project": {"_id": "$i", "name": {"$concat": ["Author ", {"$toString": "$i"}]},
                  "country": random_choice(["RO", "US", "DE", "FR", "IT"])}},
    {"$out": "authors_big"},
])

articles_big = lab["articles_big"]
authors_big = lab["authors_big"]
print(f"articles_big: {articles_big.count_documents({})} documents, authors_big: {authors_big.count_documents({})}")
show(articles_big.find().limit(2))

In [ ]:
def copy_collection(source, target_name: str, projection: dict | None = None) -> None:
    # Copy a collection into bdnv_lab with $out (falls back to a client-side copy).
    stages = [{"$project": projection}] if projection else []
    lab[target_name].drop()
    try:
        source.aggregate(stages + [{"$out": {"db": lab.name, "coll": target_name}}])
    except OperationFailure as error:
        print(f"$out to another database refused ({error.code}); copying through the client...")
        batch = []
        for document in source.aggregate(stages):
            batch.append(document)
            if len(batch) == 5_000:
                lab[target_name].insert_many(batch)
                batch = []
        if batch:
            lab[target_name].insert_many(batch)
    print(f"{lab.name}.{target_name}: {lab[target_name].estimated_document_count()} documents, "
          f"indexes: {list(lab[target_name].index_information())}")


copy_collection(mflix["movies"], "movies",
                {"title": 1, "year": 1, "type": 1, "genres": 1, "directors": 1, "cast": 1,
                 "runtime": 1, "imdb": 1, "awards": 1, "num_mflix_comments": 1})
copy_collection(mflix["comments"], "comments")
movies_copy, comments_copy = lab["movies"], lab["comments"]

### Reading a plan: `COLLSCAN` and an in-memory `SORT`

*The 20 most recent Romanian articles from 2024 onwards.* No index other than `_id` exists yet.

In [ ]:
recent_ro = {"filter": {"lang": "ro", "year": {"$gte": 2024}}, "sort": {"year": -1}, "limit": 20}

before = plan_summary(explain(articles_big, **recent_ro))

The plan reads all `N_BIG` documents (`COLLSCAN`), keeps those that satisfy the filter and sorts them in memory (`SORT`) to return 20 of them. The ratio `docsExamined / nReturned` is in the thousands.

### Adding an index

The filter contains an **equality** on `lang` and a **range** on `year`, and the result is **sorted** by `year`. An index on `{lang: 1, year: -1}` stores the keys ordered first by language and, within one language, by descending year; the plan can then jump directly to the keys `("ro", 2026)`, read them in order and stop after 20 (Course 4, slide 23).

In [ ]:
articles_big.create_index([("lang", 1), ("year", -1)])
after = plan_summary(explain(articles_big, **recent_ro))
print()
summary_table({"no index": before, "{lang: 1, year: -1}": after})

**Points to observe.**

1. The plan becomes `LIMIT <- FETCH <- IXSCAN`: the index delivers the keys already sorted, so there is no `SORT` stage, and `LIMIT` stops the scan after 20 keys.
2. `keysExamined ≈ docsExamined ≈ nReturned`: the golden rule is satisfied.
3. An index has costs: every write must update it, it occupies space (and memory, since the working set should fit in RAM), and it must be built.

### Compound indexes and the ESR rule

A compound index can serve a query only through a **prefix** of its fields (Course 4, slide 25): the index `{lang: 1, year: -1}` supports queries on `lang` and on `lang, year`, but not efficient queries on `year` alone.

When a query combines equality conditions, a sort and range conditions, the **ESR rule** (Equality, Sort, Range; Course 4, slide 26) gives the order of the fields in the index: first the fields compared for **equality**, then the **sort** fields, then the fields with **range** conditions. Consider the query

```python
find({"lang": "ro", "authorId": {"$lt": 100}}).sort({"year": -1}).limit(20)
```

with an equality on `lang`, a sort on `year` and a range on `authorId`. The next cell creates the ESR index `{lang: 1, year: -1, authorId: 1}` and, for comparison, the index `{lang: 1, authorId: 1, year: -1}` (order E, R, S), and executes the query with each of them through `hint`.

In [ ]:
esr_query = {"filter": {"lang": "ro", "authorId": {"$lt": 100}}, "sort": {"year": -1}, "limit": 20}

articles_big.create_index([("lang", 1), ("year", -1), ("authorId", 1)], name="ESR")
articles_big.create_index([("lang", 1), ("authorId", 1), ("year", -1)], name="ERS")

esr = plan_summary(explain(articles_big, **esr_query, hint="ESR"), quiet=True)
ers = plan_summary(explain(articles_big, **esr_query, hint="ERS"), quiet=True)
chosen = explain(articles_big, **esr_query, verbosity="allPlansExecution")
auto = plan_summary(chosen, quiet=True)
summary_table({"E-S-R  (hint)": esr, "E-R-S  (hint)": ers, "optimizer's choice": auto})
print("\nIndex chosen by the optimizer:", auto["indexes"],
      f"| rejected plans: {len(chosen['queryPlanner']['rejectedPlans'])}")

**Points to observe.**

1. With **E-R-S**, the range on `authorId` comes before `year` in the index, so the keys that satisfy the filter are not ordered by `year`. The plan must read **all** matching keys and sort them in memory (`SORT`), although only 20 are returned.
2. With **E-S-R**, the keys of `lang = "ro"` are already ordered by `year`. The condition on `authorId` is applied to the index keys within each year (the scan skips the keys outside the range), and the keys are read in sort order until 20 matches have been found: no in-memory sort, and the scan stops early. The number of keys examined stays close to `nReturned` and does not depend on the size of the collection.
3. With `allPlansExecution`, the output also contains the **rejected plans**, i.e. the candidates that the optimizer evaluated and discarded.

The same principle applies to pipelines: a `$match` (E, R) followed by a `$sort` (S) **at the beginning of the pipeline** is executed with the index (see below). The index `ERS` is dropped, since it is not needed in the remainder of the notebook.

In [ ]:
articles_big.drop_index("ERS")

print("Query on `year` only (not a prefix of any index):")
plan_summary(explain(articles_big, filter={"year": 2025}))
print("\nQuery on `lang` only (prefix of lang_1_year_-1 and of ESR):")
_ = plan_summary(explain(articles_big, filter={"lang": "ro"}))

### Covered queries

A query is **covered** when all the fields it filters on and returns are contained in one index: the result is produced from the index keys alone, and **no document is read** (`totalDocsExamined = 0`; Course 4, slide 28). The projection must exclude `_id` unless `_id` is part of the index.

In [ ]:
covered = plan_summary(explain(articles_big, filter={"lang": "ro", "year": 2025},
                               projection={"_id": 0, "lang": 1, "year": 1}), quiet=True)
not_covered = plan_summary(explain(articles_big, filter={"lang": "ro", "year": 2025},
                                   projection={"_id": 0, "lang": 1, "year": 1, "title": 1}), quiet=True)
summary_table({"covered (lang, year)": covered, "+ title (not in index)": not_covered})

### How a pipeline is executed

A pipeline is executed in layers (Course 4, slide 30):

1. **The prefix** of the pipeline (`$match`, `$sort`, `$limit`, `$project` at the beginning) is converted into a query that can use the **indexes** of the collection.
2. Eligible stages such as `$group` and `$lookup` are executed by the **slot-based engine (SBE)**.
3. The remaining stages process the stream document by document.

Only the stages at the beginning can use indexes. After `$unwind`, `$group` or a `$project`/`$set` with computed fields, the documents no longer correspond to the index entries. `$vectorSearch`, `$search` and `$geoNear` must be the **first** stage, since they produce the stream themselves from their own indexes.

Before execution, the optimizer **rewrites** the pipeline (Course 4, slide 31):

| Written sequence | Rewrite by the optimizer |
|---|---|
| `$set` / `$project` + `$match` | `$match` moved before, if it does not depend on the computed fields |
| `$sort` + `$limit` | top-k sort: only k documents are kept in memory |
| `$match` + `$match` | one `$match` with `$and` |
| `$limit` + `$limit`, `$skip` + `$skip` | one `$limit` (minimum) / one `$skip` (sum) |
| `$lookup` + `$unwind` (+ `$match`) | coalesced into a single `$lookup` |
| `$sort` + `$group` with `$first` | `DISTINCT_SCAN` on an index, if one exists |
| unused fields | dependency analysis: only the required fields are read |

`explain()` shows the pipeline **after** these rewrites.

#### The worked example of the handout

The pipeline adds a computed field *before* filtering, sorting and limiting. The first `explain` uses the hint `{"$natural": 1}`, which forces a collection scan and thus reproduces the situation without an index.

In [ ]:
written = [
    {"$set": {"nTags": {"$size": "$tags"}}},
    {"$match": {"lang": "ro", "year": {"$gte": 2024}}},
    {"$sort": {"year": -1}},
    {"$limit": 20},
]
print("As written, without an index:")
plan_summary(explain(articles_big, written, hint={"$natural": 1}))
print("\nAs written, with the index:")
plan_summary(explain(articles_big, written))

rewritten = [
    {"$match": {"lang": "ro", "year": {"$gte": 2024}}},
    {"$sort": {"year": -1}},
    {"$limit": 20},
    {"$set": {"nTags": {"$size": "$tags"}}},          # computed only for the 20 remaining documents
]
print("\nRewritten (match, sort, limit, then set), with the index:")
_ = plan_summary(explain(articles_big, rewritten))

**Points to observe.**

1. In the first plan, the filter of `$match` appears in the query layer (in the `COLLSCAN` stage or in `IXSCAN`), although it was written after `$set`: the optimizer moved it, because it does not depend on `nTags`.
2. In the written version, however, `$sort` and `$limit` remain *after* `$set`, outside the query layer; the sort is then performed by a pipeline stage on all matching documents.
3. In the rewritten version, `$match`, `$sort` and `$limit` form the prefix and are executed by the index (`LIMIT <- FETCH <- IXSCAN`, with about 20 keys and documents examined), and `$set` processes only the 20 remaining documents.

The lesson of the course applies: the optimizer can sometimes repair the order of the stages, but one should not rely on it. Write the stages in the right order and verify the plan with `explain()`.

#### Before and after: the order of the stages

The next cell reproduces slide 32 of the course: the same result computed with `$unwind` and `$lookup` **before** the filter, and with the filter and the sort **first**.

In [ ]:
slow = [
    {"$unwind": "$tags"},
    {"$lookup": {"from": "authors_big", "localField": "authorId", "foreignField": "_id", "as": "author"}},
    {"$match": {"lang": "ro", "year": {"$gte": 2024}}},
    {"$sort": {"year": -1, "_id": 1}},
]
fast = [
    {"$match": {"lang": "ro", "year": {"$gte": 2024}}},
    {"$sort": {"year": -1, "_id": 1}},
    {"$unwind": "$tags"},
    {"$lookup": {"from": "authors_big", "localField": "authorId", "foreignField": "_id", "as": "author"}},
]
assert list(articles_big.aggregate(slow)) == list(articles_big.aggregate(fast))

for label, pipeline in [("slow (as on slide 32)", slow), ("fast", fast)]:
    print(f"{label}:")
    plan_summary(explain(articles_big, pipeline))
    print(f"time: {timed(lambda: list(articles_big.aggregate(pipeline)))} ms\n")

**Points to observe.** The two plans are identical: the filter on `lang` and `year` depends neither on `tags` nor on `author`, so the optimizer moved it (and the sort) before `$unwind` and `$lookup`, into the query layer that uses the index. The order written by the developer did not matter here, and `explain()` is the only way to know this.

The optimizer cannot move a condition that **depends on the result of a stage**. *The 20 most recent Romanian articles whose author is from Romania*: the condition on `author.country` exists only after `$lookup`, so the join is executed for every Romanian article since 2024. Knowing the data, the developer can rewrite the query: first find the authors from Romania (a small collection), then filter the articles by `authorId` with `$in`, which uses the `ESR` index, and join only the 20 documents that are returned.

In [ ]:
join_then_filter = [
    {"$match": {"lang": "ro", "year": {"$gte": 2024}}},
    {"$lookup": {"from": "authors_big", "localField": "authorId", "foreignField": "_id", "as": "author"}},
    {"$match": {"author.country": "RO"}},                    # depends on $lookup: cannot be moved
    {"$sort": {"year": -1, "_id": 1}},
    {"$limit": 20},
]
ro_authors = authors_big.distinct("_id", {"country": "RO"})
filter_then_join = [
    {"$match": {"lang": "ro", "year": {"$gte": 2024}, "authorId": {"$in": ro_authors}}},
    {"$sort": {"year": -1, "_id": 1}},
    {"$limit": 20},
    {"$lookup": {"from": "authors_big", "localField": "authorId", "foreignField": "_id", "as": "author"}},
]
assert list(articles_big.aggregate(join_then_filter)) == list(articles_big.aggregate(filter_then_join))

for label, pipeline in [("join, then filter", join_then_filter), ("filter, then join", filter_then_join)]:
    print(f"{label}:")
    plan_summary(explain(articles_big, pipeline))
    print(f"time: {timed(lambda: list(articles_big.aggregate(pipeline)))} ms\n")

In the first plan, `EQ_LOOKUP` (or the `$lookup` stage) processes all the Romanian articles since 2024; in the second, the query layer returns only 20 documents, and the join is performed 20 times. This is the pattern of Exercise 5: filter and limit first, join last.

### `$lookup` and the index on `foreignField`

For each input document, `$lookup` searches the foreign collection for the matching documents. Without an index on `foreignField`, this search reads the foreign collection. The execution engine chooses a **join strategy**, which `explain()` reports: an *indexed loop join* if an index on `foreignField` exists, a *hash join* if the foreign collection is small enough to be loaded into memory, and otherwise a *nested loop join*, which scans the foreign collection once for every input document.

*For 100 movies of 2010: the number of comments of each.* The copy `bdnv_lab.comments` has no index on `movie_id`.

In [ ]:
movies_comments = [
    {"$match": {"year": 2010}},
    {"$limit": 100},
    {"$lookup": {"from": "comments", "localField": "_id", "foreignField": "movie_id", "as": "comments"}},
    {"$project": {"title": 1, "nComments": {"$size": "$comments"}}},
]
print("Without an index on comments.movie_id:")
no_index = plan_summary(explain(movies_copy, movies_comments))
print(f"time: {timed(lambda: list(movies_copy.aggregate(movies_comments)), repeat=1)} ms\n")

comments_copy.create_index("movie_id")
print("With the index {movie_id: 1}:")
with_index = plan_summary(explain(movies_copy, movies_comments))
print(f"time: {timed(lambda: list(movies_copy.aggregate(movies_comments)))} ms")

**Points to observe.** Without the index, the strategy is a nested loop join: the foreign collection is read for each of the 100 movies, i.e. about $100 \times 50{,}000$ documents examined. With the index, the strategy becomes an indexed loop join, and each lookup reads only the matching comments; most of the remaining documents examined come from the collection scan of `movies`, which has no index on `year`. If a join is needed by most queries, the data model itself should be reconsidered (embedding vs. referencing).

### Memory and document limits

Two limits shape the design of pipelines (Course 4, slide 35):

- **100 MB of memory per blocking stage** (`$group`, `$sort`, `$bucket`, `$setWindowFields`). Above it, the stage spills to disk (`allowDiskUse`, enabled by default since MongoDB 6.0), which is correct but slower and visible as `usedDisk: true` in `explain()`. **On M0, `allowDiskUse` is ignored and the in-memory sort limit is 32 MB: the stage fails instead.**
- **16 MB per BSON document**, which also applies to the output of `$facet` and to the arrays built by `$push`.

An unbounded `$push` is the most common way to reach both limits. The next cell measures, with `$bsonSize`, the size of the largest group document produced by `$push` and by the bounded `$topN`.

In [ ]:
for label, accumulator in [
    ("$push (unbounded)", {"$push": {"title": "$title", "text": "$text"}}),
    ("$topN (n = 3)", {"$topN": {"n": 3, "sortBy": {"year": -1}, "output": {"title": "$title", "text": "$text"}}}),
]:
    result = next(articles_big.aggregate([
        {"$group": {"_id": "$authorId", "items": accumulator}},
        {"$group": {"_id": None, "maxBytes": {"$max": {"$bsonSize": "$$ROOT"}},
                    "totalBytes": {"$sum": {"$bsonSize": "$$ROOT"}}}},
    ]))
    print(f"{label:<20} largest group: {result['maxBytes']:>8,} B   all groups: {result['totalBytes'] / 1e6:6.1f} MB")

With `N_BIG = 100,000`, each author has about 100 articles, and the groups remain small. With 100 million articles, the same `$push` would produce groups of several megabytes per author and a stage far above 100 MB. Prefer bounded accumulators (`$topN`, `$firstN`), counts (`$sum`) or a `$limit` before `$group`.

Common anti-patterns (Course 4, slide 36):

| Avoid | Prefer |
|---|---|
| `$unwind` + `$group` to count the elements of an array | `{"$size": "$tags"}` |
| a large `$skip` for pagination | range-based pagination: `{"_id": {"$gt": last_id}}` |
| `$lookup` without an index on `foreignField` | an index on `foreignField` |
| `$project` at the beginning "to make it faster" | leave it to the dependency analysis; `$project` at the end |
| `$facet` on the whole collection | an (indexed) `$match` before `$facet` |
| `$push` on millions of documents | `$topN` / `$firstN`, or `$limit` |
| returning the `embedding` field | `{"$project": {"embedding": 0}}` |

### Index maintenance: `$indexStats`, hidden indexes and the profiler

- **`$indexStats`** reports, for every index, the number of operations that used it since the server started. An index that is never used costs space and slows down writes.
- A **hidden index** is maintained but ignored by the query planner. Hiding an index before dropping it shows the effect of its removal and can be reversed instantly (`collMod` with `hidden: False`), whereas rebuilding a dropped index can take a long time on a large collection.
- The **profiler** (`setProfilingLevel`) records slow operations in the collection `system.profile`. It is not available on M0; on paid tiers, the Atlas *Query Profiler* and *Performance Advisor* provide the same information in the UI.

In [ ]:
try:
    print("Index usage since the server started:")
    for stats in articles_big.aggregate([{"$indexStats": {}}]):
        print(f"  {stats['name']:<16} ops = {stats['accesses']['ops']}")
except OperationFailure as error:
    print(f"$indexStats unavailable on this cluster: {error.details.get('errmsg', error)}")


def set_hidden(name: str, hidden: bool) -> None:
    lab.command({"collMod": "articles_big", "index": {"name": name, "hidden": hidden}})


# Hide the indexes that serve the query and observe its plan
try:
    for name in ("lang_1_year_-1", "ESR"):
        set_hidden(name, True)
    print("\nWith the indexes hidden:")
    plan_summary(explain(articles_big, **recent_ro))
finally:
    for name in ("lang_1_year_-1", "ESR"):
        try:
            set_hidden(name, False)
        except OperationFailure as error:
            print(f"collMod unavailable on this cluster: {error.details.get('errmsg', error)}")
            break
print("\nVisible again:")
plan_summary(explain(articles_big, **recent_ro))

try:
    lab.command("profile", 1, slowms=50)          # level 1: operations slower than 50 ms
    list(articles_big.aggregate(slow))
    show(lab["system.profile"].find({}, {"op": 1, "ns": 1, "millis": 1, "planSummary": 1}).sort("ts", -1).limit(3))
    lab.command("profile", 0)
except OperationFailure as error:
    print(f"\nProfiler unavailable on this cluster: {error.details.get('errmsg', error)}")

### Checklist for slow pipelines

Course 4 (slide 40) summarizes the procedure:

1. `explain("executionStats")` on the pipeline;
2. `IXSCAN` in the prefix, not `COLLSCAN`;
3. `totalKeysExamined ≈ totalDocsExamined ≈ nReturned`;
4. no in-memory `SORT`, `usedDisk: false`;
5. `$match` and `$sort` as early as possible (ESR);
6. an index on `foreignField` for every `$lookup`;
7. `$indexStats`: remove the indexes that are never used;
8. run `explain()` again after every change.

### Exercises for Part II

#### Exercise 5: Find the problems (handout, Exercise 2) *(homework)*

*Objective: to diagnose and repair a pipeline with `explain()`, the ESR rule and the limits of Part II.*

The starter cell below generates `orders` (100,000 documents, each with an array `items`) and `customers` (20,000 documents with the key `custId`); the only index of both collections is `_id`. The volumes are smaller than in the handout (5 million and 200,000), and the `$skip` has been reduced accordingly.

```python
bad_pipeline = [
    {"$lookup": {"from": "customers", "localField": "customerId", "foreignField": "custId", "as": "c"}},
    {"$unwind": "$items"},
    {"$match": {"status": "shipped", "orderDate": {"$gte": datetime(2026, 1, 1)}}},
    {"$group": {"_id": "$customerId", "items": {"$push": "$items"}, "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
    {"$skip": 5000}, {"$limit": 20},
]
```

1. Identify at least four performance or correctness problems. **Do not execute the original pipeline on the full collection**: inspect it with `explain(orders, bad_pipeline, verbosity="queryPlanner")`, which does not execute it, and, if you want to observe its cost, execute it with `maxTimeMS=15000`.
2. Rewrite the pipeline. The result must remain the same for the customers that are returned. Use range-based pagination instead of `$skip`.
3. Which index do you create on `orders`, and which on `customers`? Justify the choice with the ESR rule.
4. Which values do you expect in `explain("executionStats")` after the optimization? Verify them with `plan_summary`.
5. Which limit (100 MB per stage or 16 MB per document) can the `$push` reach, and how is it avoided?

#### Exercise 6: Index design on `listingsAndReviews` *(in the lab, ~15 min)*

*Objective: to apply the ESR rule and to compare candidate indexes with measurements.*

The starter cell copies `listingsAndReviews` (without `reviews` and `images`) into `bdnv_lab.listings`. Consider the query

```python
find({"address.market": "Porto", "bedrooms": {"$gte": 2}}).sort({"review_scores.review_scores_rating": -1}).limit(10)
```

1. Which index does the ESR rule recommend? Create it, together with at least two alternative orders of the same fields.
2. For each index (with `hint`) and without an index, record `plan`, `nReturned`, `keysExamined`, `docsExamined` and the presence of `SORT` in a table (`summary_table`). Which index does the optimizer choose without a hint?
3. Write a covered variant of the query that returns only the market, the number of bedrooms and the rating. Which index does it need, and what is `docsExamined`?
4. Hide the ESR index and execute the query again. Report what changes in the plan, then make the index visible again.

#### Exercise 7: Multi-planning and the plan cache *(homework)*

*Objective: to observe how the optimizer chooses between candidate plans.*

1. With the indexes `lang_1_year_-1` and `ESR` on `articles_big`, execute `explain(..., verbosity="allPlansExecution")` for `find({"lang": "ro", "year": 2025, "authorId": {"$lt": 10}})`. How many candidate plans are evaluated, and which metric distinguishes the winning plan from the rejected ones?
2. Execute the queries `{"lang": "ro", "year": 2025}` and `{"lang": "fr", "year": 2018}`. Compare their `queryHash` / `planCacheShapeHash` in the output of `explain`. What does the equality of these values mean?
3. Create an index `{year: 1}` and execute `find({"year": {"$gte": 2016}, "lang": "ro"})`. Which index is chosen, and why? (Hint: compare the selectivity of the two conditions.)

In [ ]:
# Starter cell for Exercise 5: orders and customers, generated on the server
from datetime import datetime

N_ORDERS, N_CUSTOMERS = 100_000, 20_000
start_date, end_date = datetime(2024, 1, 1), datetime(2026, 11, 1)
span_ms = int((end_date - start_date).total_seconds() * 1000)

lab["customers"].drop()
lab.aggregate([
    {"$documents": [{"i": {"$range": [0, N_CUSTOMERS]}}]},
    {"$unwind": "$i"},
    {"$project": {"_id": "$i", "custId": {"$concat": ["C", {"$toString": "$i"}]},
                  "name": {"$concat": ["Customer ", {"$toString": "$i"}]},
                  "country": random_choice(["RO", "US", "DE", "FR", "IT"])}},
    {"$out": "customers"},
])
lab["orders"].drop()
lab.aggregate([
    {"$documents": [{"i": {"$range": [0, N_ORDERS]}}]},
    {"$unwind": "$i"},
    {"$project": {
        "_id": "$i",
        "customerId": {"$concat": ["C", {"$toString": random_int(0, N_CUSTOMERS)}]},
        "status": random_choice(["shipped", "shipped", "pending", "cancelled"]),
        "orderDate": {"$add": [start_date, random_int(0, span_ms)]},
        "items": {"$map": {"input": {"$range": [0, random_int(1, 5)]},
                           "in": {"sku": {"$concat": ["SKU-", {"$toString": random_int(0, 500)}]},
                                  "qty": random_int(1, 4),
                                  "price": random_int(5, 200)}}},
    }},
    {"$out": "orders"},
])
orders, customers = lab["orders"], lab["customers"]
print(f"orders: {orders.count_documents({})}, customers: {customers.count_documents({})}")
show(orders.find().limit(1))

bad_pipeline = [
    {"$lookup": {"from": "customers", "localField": "customerId", "foreignField": "custId", "as": "c"}},
    {"$unwind": "$items"},
    {"$match": {"status": "shipped", "orderDate": {"$gte": datetime(2026, 1, 1)}}},
    {"$group": {"_id": "$customerId", "items": {"$push": "$items"}, "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
    {"$skip": 5000}, {"$limit": 20},
]
_ = plan_summary(explain(orders, bad_pipeline, verbosity="queryPlanner"))

In [ ]:
# Starter cell for Exercise 6: a copy of listingsAndReviews without the large fields
copy_collection(airbnb, "listings", {"reviews": 0, "images": 0})
listings = lab["listings"]

ex5_query = {"filter": {"address.market": "Porto", "bedrooms": {"$gte": 2}},
             "sort": {"review_scores.review_scores_rating": -1}, "limit": 10}
_ = plan_summary(explain(listings, **ex5_query))

## Cleaning up

The scratch database occupies part of the 512 MB of an M0 cluster. Set `CLEAN_UP = True` when you have finished the exercises. Lab 5 recreates the collections it needs, so dropping the database does not affect it. The sample databases are not affected.

In [ ]:
CLEAN_UP = False

if CLEAN_UP:
    client.drop_database(lab.name)
    print(f"Dropped database {lab.name}")

### Summary of the key concepts

| Concept | Role in this laboratory |
|---|---|
| Document, collection, BSON | Embedded documents and arrays instead of joins; `_id`; `Decimal128`, `Date`, `ObjectId` |
| CRUD | `insert_*`, `find` (filter, projection, sort, limit), `update_*` with `$set`/`$inc`/`$push`, `delete_*` |
| Query operators | Comparisons, `$in`, logical operators, dot notation, conditions on arrays |
| Aggregation pipeline | A stream of documents through stages; the order of the stages determines the result and the cost |
| `$match` before / after `$group` | `WHERE` / `HAVING` |
| `$unwind`, `$size` | One document per array element; the size of an array without unwinding |
| `$lookup` | Left outer join (equality or sub-pipeline); group or limit first, join last |
| `$facet`, `$bucket`, `$setWindowFields` | Several aggregations of one input; ranges; window functions |
| Query plan, `explain()` | `queryPlanner` (not executed), `executionStats`, `allPlansExecution` |
| `COLLSCAN`, `IXSCAN`, `FETCH`, `SORT`, `PROJECTION_COVERED` | The plan stages to look for; `keys ≈ docs ≈ nReturned` |
| Compound index, prefix, ESR | Equality, Sort, Range: no in-memory sort, early termination |
| Covered query | Answered from the index, `totalDocsExamined = 0` |
| Pipeline rewrites | `$match` moved forward, `$sort` + `$limit` as top-k; verified with `explain()` |
| `$lookup` strategies | Indexed loop join (index on `foreignField`), hash join, nested loop join |
| Limits | 100 MB per blocking stage, 16 MB per document; M0 without spilling to disk |
| `$indexStats`, hidden indexes, profiler | Index maintenance |

## References

**Official documentation**
- MongoDB. *Get Started with Atlas*. https://www.mongodb.com/docs/atlas/getting-started/
- MongoDB. *Load Sample Data*; *Sample AirBnB Listings Dataset*; *Sample Mflix Dataset*. https://www.mongodb.com/docs/atlas/sample-data/
- MongoDB. *Documents*, *Databases and Collections*, *BSON Types*. MongoDB Manual. https://www.mongodb.com/docs/manual/core/document/
- MongoDB. *MongoDB CRUD Operations* and *Query and Projection Operators*. MongoDB Manual. https://www.mongodb.com/docs/manual/crud/, https://www.mongodb.com/docs/manual/reference/operator/query/
- MongoDB. *SQL to MongoDB Mapping Chart*. MongoDB Manual. https://www.mongodb.com/docs/manual/reference/sql-comparison/
- MongoDB. *Data Modeling* (embedding vs. references). MongoDB Manual. https://www.mongodb.com/docs/manual/data-modeling/
- MongoDB. *Aggregation Operations*; *Aggregation Pipeline Optimization*; *Aggregation Pipeline Limits*. MongoDB Manual. https://www.mongodb.com/docs/manual/aggregation/
- MongoDB. *Query Optimization*; *Explain Results*; *The ESR (Equality, Sort, Range) Guideline*; *Hidden Indexes*. MongoDB Manual. https://www.mongodb.com/docs/manual/core/query-optimization/
- MongoDB. *MongoDB 8.3 Release Notes*. https://www.mongodb.com/docs/manual/release-notes/8.3/
- MongoDB. *Atlas Free Cluster Limits*. https://www.mongodb.com/docs/atlas/reference/free-shared-limitations/
- PyMongo. *PyMongo Documentation*. https://pymongo.readthedocs.io/

**Course materials**
- *BDNV, Course 4: MongoDB: Aggregation Pipelines, Optimizare & Vector Search*, and the handout of Course 4.
- I. Banu. *Lab 1: LangChain Introduction*, *Lab 2: Tools and Short-Term Memory*, *Lab 3: Middleware and Agent Customization*.

**Courses**
- MongoDB University. *Introduction to MongoDB*, *Indexing Design Fundamentals*, *Query Optimization*. https://learn.mongodb.com

> **Note on the preparation of this document.** Parts of this laboratory guide were drafted with the assistance of Claude (Anthropic), an AI assistant.